<a href="https://colab.research.google.com/github/yamms2340/researchWorkCodes/blob/main/TrainCnnModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell first to connect to your Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

# 1. Define the Network Architecture
class Cnn1d(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5)
        self.c2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, dilation=2)
        self.r = nn.ReLU()
        self.fc = nn.Linear(32 * 992, 3)

    def forward(self, x):
        x = self.r(self.c1(x))
        x = self.r(self.c2(x))
        x = x.view(x.shape[0], -1)
        return self.fc(x)

# 2. Load the Training Data from Drive
print("Loading dataset...")
df_data = pd.read_csv("/content/drive/MyDrive/research_Final/lorenz_data.csv")
z_data = df_data['z_var'].values
z_norm = (z_data - np.min(z_data)) / (np.max(z_data) - np.min(z_data) + 1e-8)
train_tensor = torch.tensor(z_norm, dtype=torch.float32).view(1, 1, -1)

# 3. Load the Target Labels from Drive
print("Loading classical ground truth targets...")
df_targets = pd.read_csv("/content/drive/MyDrive/research_Final/classical_results.csv")
true_le = df_targets['Classical_Value'].values
target_tensor = torch.tensor(true_le, dtype=torch.float32).view(1, -1)

# 4. Initialize Model, Optimizer, and Loss Function
model = Cnn1d()
optimizer = optim.Adam(model.parameters(), lr=0.001)
huber_loss = nn.HuberLoss(delta=1.0)

epochs = 100
print("\n--- Starting Training Loop ---")

# 5. The Training Loop
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    predictions = model(train_tensor)
    loss = huber_loss(predictions, target_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.6f}")

# 6. Save the Trained Weights to Drive
weights_filename = "/content/drive/MyDrive/research_Final/cnn_weights.pth"
torch.save(model.state_dict(), weights_filename)
print(f"\nTraining Complete! Model weights successfully saved to: {weights_filename}")

Mounted at /content/drive
Loading dataset...
Loading classical ground truth targets...

--- Starting Training Loop ---
Epoch [10/100] | Loss: 0.741987
Epoch [20/100] | Loss: 0.447925
Epoch [30/100] | Loss: 0.019198
Epoch [40/100] | Loss: 0.035022
Epoch [50/100] | Loss: 0.022478
Epoch [60/100] | Loss: 0.005038
Epoch [70/100] | Loss: 0.000690
Epoch [80/100] | Loss: 0.000113
Epoch [90/100] | Loss: 0.000074
Epoch [100/100] | Loss: 0.000043

Training Complete! Model weights successfully saved to: /content/drive/MyDrive/research_Final/cnn_weights.pth
